In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from datasets import load_dataset
from collections import Counter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import numpy as np

c:\Vasu\Guvi\Sentiment Analysis - FP\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [3]:
dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data = dataset["test"]

print(len(train_data), len(test_data))

25000 25000


In [4]:
train_data = train_data.shuffle(seed=42).select(range(5000))
test_data = test_data.shuffle(seed=42).select(range(2000))

## Build Vocabulary

In [5]:
MAX_VOCAB_SIZE = 20000
MAX_SEQ_LEN = 300

def tokenize(text):
    return text.lower().split()

counter = Counter()

for text in train_data["text"]:
    counter.update(tokenize(text))

vocab = {"<pad>": 0, "<unk>": 1}

for word, _ in counter.most_common(MAX_VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

print("Vocab size:", len(vocab))

Vocab size: 20000


## Encoding Function

In [6]:
def encode(text):
    tokens = tokenize(text)
    ids = [vocab.get(token, vocab["<unk>"]) for token in tokens[:MAX_SEQ_LEN]]
    return torch.tensor(ids)

## Create Dataset Class

In [7]:
class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset):
        self.texts = hf_dataset["text"]
        self.labels = hf_dataset["label"]
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        text_tensor = encode(self.texts[idx])
        label = torch.tensor(self.labels[idx]).float()
        return text_tensor, label

## Collate Function (Padding)

In [8]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    texts = [item[0] for item in batch]
    labels = torch.stack([item[1] for item in batch])
    
    texts = pad_sequence(texts, batch_first=True)
    return texts, labels

## DataLoaders

In [9]:
BATCH_SIZE = 64

train_dataset = IMDBDataset(train_data)
test_dataset = IMDBDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

## Build Custom LSTM Model

In [10]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        out = self.fc(hidden[-1])
        return self.sigmoid(out).squeeze()

## Initialize Model

In [11]:
model = SentimentLSTM(len(vocab)).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [12]:
from collections import Counter

print("Train labels:", Counter(train_data["label"]))
print("Test labels:", Counter(test_data["label"]))

Train labels: Counter({1: 2506, 0: 2494})
Test labels: Counter({1: 1000, 0: 1000})


## Training Loop

In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for texts, labels in train_loader:
        texts, labels = texts.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

## Evaluation

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for texts, labels in test_loader:
        texts = texts.to(device)
        outputs = model(texts)
        preds = (outputs > 0.5).cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("Precision:", precision_score(all_labels, all_preds))
print("Recall:", recall_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))

Accuracy: 1.0
Precision: 0.0
Recall: 0.0
F1: 0.0


c:\Vasu\Guvi\Sentiment Analysis - FP\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Vasu\Guvi\Sentiment Analysis - FP\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Vasu\Guvi\Sentiment Analysis - FP\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
